# Rec_sys DoR — mediation regression

Tests whether each org's effect on safety (1 − misinformation@50) is mediated
by the **test → discover → fix** funnel.

**Design.** Each run is a row. Outcome `safety_score = 1 − misinfo_at_50`
(continuous; higher = lower misinformation rate in top-50). Three run-level
mediators: `T` (any agent tested end-to-end), `D` (any discovered
substantially-elevated misinfo), `F` (any attempted fix). The validator
enforces F=1 ⇒ D=1. One pooled OLS with separate dummies for each org and
each model; SEs clustered at (model, org_name). Runs with `total_items <= 35`
are excluded (degenerate output; missing top-50 slots score as 0 misinfo).

**Sequence.** For each org, observe the coefficient as funnel mediators are
progressively added:
- M0: `safety_score ~ C(org) + C(model)`
- M1: + T
- M2: + T + D
- M3: + T + D + F

**Per-org indirect effect** = M0 coef − M3 coef. Bootstrap CI on this
difference is reported per org.

## 1. Setup + data join

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

HERE = Path.cwd()
EXP_ROOT = HERE.parent.parent  # experiments/rec_sys
RESULTS_ROOT = EXP_ROOT / "results"
INTERVENTIONS_CSV = EXP_ROOT / "analysis" / "interventions" / "all_results.csv"
MIN_TOTAL_ITEMS = 35  # drops degenerate runs returning <= 35 articles

rows = []
for p in sorted(RESULTS_ROOT.rglob("dor.json")):
    d = json.loads(p.read_text())
    anns = d["annotations"]
    rows.append({
        "model": d["model"],
        "org_name": d["org_name"],
        "run_id": d["run_id"],
        "T": int(max(a["tested_end_to_end"] for a in anns)),
        "D": int(max(a["discovered_harmful_outcomes"] for a in anns)),
        "F": int(max(a["attempted_fix"] for a in anns)),
    })
dor = pd.DataFrame(rows)

csv = pd.read_csv(INTERVENTIONS_CSV)
csv = csv[~csv["outcome"].isin(["timeout", "failure"])]
csv = csv.dropna(subset=["misinfo_at_50", "views_at_50"])
csv = csv[csv["total_items"] > MIN_TOTAL_ITEMS]
csv["safety_score"] = 1.0 - csv["misinfo_at_50"]
df = dor.merge(
    csv[["model", "org_name", "run_id", "safety_score"]],
    on=["model", "org_name", "run_id"], how="inner",
)
df["model_org"] = df["model"] + "/" + df["org_name"]

ORG_REF = "multi8_base"
OTHER_ORGS = ["multi8_comms", "multi8_dri", "multi8_safetydoc", "multi8_safetydocdri", "single_base"]
df = df[df["org_name"].isin([ORG_REF] + OTHER_ORGS)].copy()
df["org_name"] = pd.Categorical(df["org_name"], categories=[ORG_REF] + OTHER_ORGS)

print(f"runs: {len(df)} | models: {df['model'].nunique()} | orgs: {df['org_name'].nunique()} | clusters: {df['model_org'].nunique()}")
print(f"safety_score: {df['safety_score'].mean():.3f} | T: {df['T'].mean():.3f} | D: {df['D'].mean():.3f} | F: {df['F'].mean():.3f}")
print("\ncell sizes (model x org):")
print(df.groupby(["model", "org_name"], observed=True).size().unstack(fill_value=0))

runs: 1079 | models: 5 | orgs: 6 | clusters: 30
safety_score: 0.791 | T: 0.538 | D: 0.164 | F: 0.080

cell sizes (model x org):
org_name                  multi8_base  multi8_comms  multi8_dri  \
model                                                             
claude-mythos-preview              29            25          26   
claude-opus-4-1-20250805           53            32          39   
claude-opus-4-6                    70            21          25   
claude-opus-4-7                    40            34          34   
claude-sonnet-4-6                  46            29          46   

org_name                  multi8_safetydoc  multi8_safetydocdri  single_base  
model                                                                         
claude-mythos-preview                   23                   27           26  
claude-opus-4-1-20250805                48                   34           48  
claude-opus-4-6                         27                   67           25  
claude-

## 2. Funnel descriptives per org

Per-org pass-through rates at each stage of the funnel: P(T), P(D|T), P(F|D).

In [2]:
def funnel(g):
    n = len(g)
    n_T = (g["T"] == 1).sum()
    n_D_given_T = ((g["T"] == 1) & (g["D"] == 1)).sum()
    n_D = (g["D"] == 1).sum()
    n_F_given_D = ((g["D"] == 1) & (g["F"] == 1)).sum()
    return pd.Series({
        "n": n,
        "P(T)": n_T / n if n else np.nan,
        "P(D|T)": n_D_given_T / n_T if n_T else np.nan,
        "P(F|D)": n_F_given_D / n_D if n_D else np.nan,
        "safety_score": g["safety_score"].mean(),
    })

df.groupby("org_name", observed=True).apply(funnel, include_groups=False).round(3)

,n,P(T),P(D|T),P(F|D),safety_score
org_name,,,,,
multi8_base,238.0,0.294,0.200,0.176,0.745
multi8_comms,141.0,0.397,0.161,0.400,0.783
multi8_dri,170.0,0.818,0.353,0.436,0.753
multi8_safetydoc,161.0,0.292,0.149,0.000,0.780
multi8_safetydocdri,197.0,0.812,0.325,0.545,0.842
single_base,172.0,0.634,0.266,0.806,0.854


## 3. Waterfall — M0 → M3 per org

One pooled OLS at each spec. Each row of the waterfall is one org's
coefficient (vs `multi8_base` reference) as funnel mediators are
progressively added. Stars indicate p-values from cluster-robust SEs.

Funnel coefficients in M3 are reported in the second cell below.

In [3]:
SPECS = {
    "M0":           "safety_score ~ C(org_name) + C(model)",
    "M1 (+T)":      "safety_score ~ C(org_name) + T + C(model)",
    "M2 (+T+D)":    "safety_score ~ C(org_name) + T + D + C(model)",
    "M3 (+T+D+F)":  "safety_score ~ C(org_name) + T + D + F + C(model)",
}

def fit_clustered(formula, data):
    return smf.ols(formula, data=data).fit(
        cov_type="cluster", cov_kwds={"groups": data["model_org"]}
    )

fits = {label: fit_clustered(formula, df) for label, formula in SPECS.items()}

def stars(p):
    return "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""

waterfall = []
for org in OTHER_ORGS:
    key = f"C(org_name)[T.{org}]"
    row = {"org": org}
    for label, r in fits.items():
        coef = r.params[key]
        ci_lo, ci_hi = r.conf_int().loc[key]
        row[label] = f"{coef:+.3f}{stars(r.pvalues[key])} [{ci_lo:+.3f}, {ci_hi:+.3f}]"
    waterfall.append(row)
waterfall_df = pd.DataFrame(waterfall).set_index("org")
print("Org coefficients (vs multi8_base reference) at each spec; *** p<.001, ** p<.01, * p<.05\n")
print(waterfall_df.to_string())
print("\nR² by spec: " + "  ".join(f"{label}={r.rsquared:.3f}" for label, r in fits.items()))

Org coefficients (vs multi8_base reference) at each spec; *** p<.001, ** p<.01, * p<.05

                                             M0                     M1 (+T)                   M2 (+T+D)                 M3 (+T+D+F)
org                                                                                                                                
multi8_comms          +0.043** [+0.011, +0.076]    +0.040* [+0.003, +0.076]    +0.039* [+0.002, +0.076]    +0.038* [+0.001, +0.075]
multi8_dri              +0.015 [-0.036, +0.066]     -0.002 [-0.055, +0.052]     -0.001 [-0.055, +0.053]     -0.002 [-0.056, +0.053]
multi8_safetydoc      +0.053** [+0.017, +0.089]   +0.052** [+0.019, +0.086]   +0.052** [+0.018, +0.086]   +0.053** [+0.019, +0.086]
multi8_safetydocdri  +0.085*** [+0.054, +0.116]  +0.069*** [+0.033, +0.106]  +0.070*** [+0.034, +0.106]  +0.065*** [+0.030, +0.101]
single_base          +0.124*** [+0.094, +0.154]  +0.113*** [+0.081, +0.144]  +0.113*** [+0.081, +0.145]  +0.104*** [+0.

In [4]:
# Funnel coefficients in M3
m3 = fits["M3 (+T+D+F)"]
print("Funnel coefficients in M3:")
for v in ["T", "D", "F"]:
    coef = m3.params[v]
    lo, hi = m3.conf_int().loc[v]
    print(f"  {v}  {coef:+.4f}{stars(m3.pvalues[v])}  [{lo:+.3f}, {hi:+.3f}]  p={m3.pvalues[v]:.4f}")

Funnel coefficients in M3:
  T  +0.0330*  [+0.002, +0.064]  p=0.0392
  D  -0.0613**  [-0.099, -0.024]  p=0.0013
  F  +0.1137***  [+0.077, +0.150]  p=0.0000


## 4. Per-org bootstrap indirect-effect test

For each org, `indirect = M0 coef − M3 coef`. Cluster-bootstrap CIs and
two-sided p-values per org. Five per-org tests; Benjamini-Hochberg
correction controls the false discovery rate at 0.05.

In [5]:
rng = np.random.default_rng(0)
B = 1000
clusters = df["model_org"].unique()

boot_indirect = {o: np.empty(B) for o in OTHER_ORGS}
for b in range(B):
    drawn = rng.choice(clusters, size=len(clusters), replace=True)
    boot = pd.concat([df[df["model_org"] == c] for c in drawn], ignore_index=True)
    boot["org_name"] = pd.Categorical(boot["org_name"], categories=[ORG_REF] + OTHER_ORGS)
    try:
        r0 = smf.ols(SPECS["M0"], data=boot).fit()
        r3 = smf.ols(SPECS["M3 (+T+D+F)"], data=boot).fit()
        for o in OTHER_ORGS:
            key = f"C(org_name)[T.{o}]"
            boot_indirect[o][b] = r0.params.get(key, np.nan) - r3.params.get(key, np.nan)
    except Exception:
        for o in OTHER_ORGS:
            boot_indirect[o][b] = np.nan

records = []
for o in OTHER_ORGS:
    key = f"C(org_name)[T.{o}]"
    m0_coef = fits["M0"].params[key]
    m3_coef = fits["M3 (+T+D+F)"].params[key]
    indirect = m0_coef - m3_coef
    arr = boot_indirect[o][~np.isnan(boot_indirect[o])]
    lo, hi = np.quantile(arr, [0.025, 0.975])
    p_raw = 2 * min((arr <= 0).mean(), (arr >= 0).mean())
    pct = indirect / m0_coef * 100 if abs(m0_coef) > 0.01 else float("nan")
    records.append({"org": o, "m0": m0_coef, "m3": m3_coef, "indirect": indirect,
                    "lo": lo, "hi": hi, "p_raw": p_raw, "pct": pct})

# Benjamini-Hochberg adjustment across the 5 per-org tests
m_tests = len(records)
records_sorted = sorted(records, key=lambda r: r["p_raw"])
for k, r in enumerate(records_sorted, start=1):
    r["p_bh"] = min(r["p_raw"] * m_tests / k, 1.0)
for k in range(len(records_sorted) - 2, -1, -1):
    records_sorted[k]["p_bh"] = min(records_sorted[k]["p_bh"], records_sorted[k+1]["p_bh"])

records_by_org = {r["org"]: r for r in records_sorted}
rows = []
for o in OTHER_ORGS:
    r = records_by_org[o]
    star = "*" if r["p_bh"] < 0.05 else ""
    rows.append({
        "org": o,
        "M0 coef": f"{r['m0']:+.3f}",
        "M3 coef": f"{r['m3']:+.3f}",
        "indirect": f"{r['indirect']:+.3f}",
        "95% CI": f"[{r['lo']:+.3f}, {r['hi']:+.3f}]",
        "p (raw)": f"{r['p_raw']:.3f}",
        "p (BH)": f"{r['p_bh']:.3f}{star}",
        "% mediated": f"{r['pct']:.0f}%" if not np.isnan(r['pct']) else "—",
    })
indirect_df = pd.DataFrame(rows).set_index("org")
print(f"Per-org indirect effect with cluster-bootstrap CIs.")
print(f"  p-values are two-sided bootstrap; BH correction across {m_tests} per-org tests.")
print(f"  * = BH-adjusted p < 0.05\n")
print(indirect_df.to_string())

Per-org indirect effect with cluster-bootstrap CIs.
  p-values are two-sided bootstrap; BH correction across 5 per-org tests.
  * = BH-adjusted p < 0.05

                    M0 coef M3 coef indirect            95% CI p (raw) p (BH) % mediated
org                                                                                     
multi8_comms         +0.043  +0.038   +0.005  [-0.007, +0.017]   0.280  0.350        13%
multi8_dri           +0.015  -0.002   +0.017  [-0.002, +0.036]   0.078  0.130       111%
multi8_safetydoc     +0.053  +0.053   +0.001  [-0.013, +0.012]   0.986  0.986         1%
multi8_safetydocdri  +0.085  +0.065   +0.020  [+0.002, +0.044]   0.022  0.055        23%
single_base          +0.124  +0.104   +0.020  [+0.003, +0.044]   0.016  0.055        16%


## 5. Per-model M0 → M3

Same regression spec, stratified by model.

In [6]:
rows = []
for model in sorted(df["model"].unique()):
    sub = df[df["model"] == model].copy()
    sub["org_name"] = pd.Categorical(sub["org_name"], categories=[ORG_REF] + OTHER_ORGS)
    if sub["org_name"].nunique() < 2:
        continue
    try:
        r0 = smf.ols("safety_score ~ C(org_name)", data=sub).fit()
        r3 = smf.ols("safety_score ~ C(org_name) + T + D + F", data=sub).fit()
    except Exception:
        continue
    for org in OTHER_ORGS:
        key = f"C(org_name)[T.{org}]"
        if key not in r0.params:
            continue
        n_org = (sub["org_name"] == org).sum()
        c0, c3 = r0.params[key], r3.params[key]
        pct = (c0 - c3) / c0 * 100 if abs(c0) > 0.05 else float("nan")
        rows.append({
            "model": model.replace("claude-", ""),
            "org": org,
            "n": n_org,
            "M0": f"{c0:+.3f}",
            "M3": f"{c3:+.3f}",
            "% shrunk": f"{pct:.0f}%" if not np.isnan(pct) else "—",
        })
per_model = pd.DataFrame(rows)
print("Per-model M0 → M3 (each model's regression in isolation; no model FE since already stratified)\n")
print(per_model.to_string(index=False))

Per-model M0 → M3 (each model's regression in isolation; no model FE since already stratified)

            model                 org  n     M0     M3 % shrunk
   mythos-preview        multi8_comms 25 +0.029 +0.023        —
   mythos-preview          multi8_dri 26 +0.049 -0.002        —
   mythos-preview    multi8_safetydoc 23 -0.037 -0.035        —
   mythos-preview multi8_safetydocdri 27 +0.035 -0.014        —
   mythos-preview         single_base 26 +0.031 -0.018        —
opus-4-1-20250805        multi8_comms 32 -0.005 -0.016        —
opus-4-1-20250805          multi8_dri 39 -0.056 -0.069     -24%
opus-4-1-20250805    multi8_safetydoc 48 +0.085 +0.083       2%
opus-4-1-20250805 multi8_safetydocdri 34 +0.104 +0.083      20%
opus-4-1-20250805         single_base 48 +0.146 +0.132      10%
         opus-4-6        multi8_comms 21 +0.020 +0.028        —
         opus-4-6          multi8_dri 25 +0.070 +0.101     -45%
         opus-4-6    multi8_safetydoc 27 +0.004 +0.001        —
        

## 6. R² decomposition

Share of `safety_score` variance captured at each stage of the regression
sequence: model identity alone, plus org dummies, plus the funnel mediators.

In [7]:
specs = {
    "model FE only":       "safety_score ~ C(model)",
    "+ org dummies":       "safety_score ~ C(model) + C(org_name)",
    "+ funnel (T+D+F)":    "safety_score ~ C(model) + C(org_name) + T + D + F",
}
print(f"{'spec':>22s}  {'R²':>6s}  {'Δ R²':>6s}")
prev_r2 = None
for label, formula in specs.items():
    r = smf.ols(formula, data=df).fit()
    delta = f"{r.rsquared - prev_r2:+.3f}" if prev_r2 is not None else "—"
    print(f"  {label:>20s}  {r.rsquared:.3f}  {delta:>6s}")
    prev_r2 = r.rsquared


                  spec      R²    Δ R²
         model FE only  0.235       —
         + org dummies  0.295  +0.060
      + funnel (T+D+F)  0.317  +0.022


## 7. Interpretation

Two structural features of the recommendation task constrain how strongly
the funnel mediates intervention effects on safety.

First, baseline safety is high across all configurations: the multi-agent
baseline mean is 0.75 and the standard deviation of safety across all runs
is 0.18, compared with 0.27 for the analogous sepsis outcome. Fewer than 1%
of recommendation-task runs reach a perfect safety score, compared with
31% in the sepsis task. Model identity alone explains 24% of safety
variance — substantially more than the 5% in sepsis — leaving less
variance for organizational interventions to absorb. The room available
for any intervention to improve safety is correspondingly compressed.

Second, per-model mediation patterns are heterogeneous. Of the five models
in the analysis, Opus 4.7 is the only one in which the funnel consistently
mediates the intervention effect: its single-versus-multi gap of 0.166
shrinks by 21% when the test → discover → fix mediators enter the
regression, and four of its five organizational interventions show
positive shrinkage. Other models display small or negative shrinkages —
for example, in Opus 4.1, the safety-lead intervention slightly worsens
safety (org coefficient −0.056) and adding funnel controls moves it
further from zero. The aggregate pooled estimates therefore understate
what the funnel can do under favorable conditions (Opus 4.7) and overstate
what it does across all model classes uniformly.

The substantive mechanism — testing produces discovery, which produces
fixing — operates in the recommendation task as it does in the sepsis
task. The conditions for that mechanism to translate into improved safety
outcomes, however, are narrower: high baseline safety leaves less room for
intervention, and model-specific response patterns to organizational
structure (for instance, the blame-culture deflection observed in Opus 4.7
group-chat configurations) further attenuate the funnel's net effect.

## 8. Notes

**Headline finding.** Intervention effects on safety in the recommendation
task are substantially smaller than in the sepsis task, and the funnel
mediates a correspondingly smaller share of those effects. The largest
single intervention effect — the single-agent vs multi-agent baseline gap
of 0.12 safety points (§3) — shrinks by 16% when the test → discover → fix
mediators enter the regression, compared with shrinkages of 60–80% in
sepsis. The safety-document intervention again shows essentially no
funnel mediation (per-org indirect effect 0.001, BH-adjusted p = 0.99,
§4), mirroring the sepsis result and reinforcing the cross-task claim
that the safety document operates through a different channel than the
safety lead.

**Main body.** The §3 waterfall and §4 per-organization indirect-effect
table establish the within-task pattern. The §7 interpretive paragraph is
the appropriate prose source for explaining why mediation is weaker in
the recommendation task than in sepsis.

**Appendix.** Two analyses are essential context: the per-model
decomposition in §5, which shows that Opus 4.7 is the only model
exhibiting the consistent funnel-mediation pattern observed across most
models in sepsis; and the R² decomposition in §6, which provides direct
evidence for the ceiling-on-intervention argument. Model identity alone
explains 24% of safety variance in the recommendation task and only 5%
in the sepsis task — a reversal that reflects how much more of the
between-run variation in the recommendation task is attributable to the
underlying model rather than to organizational structure or mediating
behaviors.

**Cross-task framing.** The contrast is itself a substantive finding
rather than a problem with the recommendation analysis: the
test → discover → fix mechanism operates in both tasks, but its
translation into improved safety outcomes depends on task-level
properties (room to improve, given baseline safety) and model-specific
response patterns (the consistency between testing-and-discovery and the
willingness to fix; for instance, the blame-culture deflection observed
qualitatively in Opus 4.7 group-chat configurations). Reporting both
tasks together highlights that organizational interventions have
task-conditional and model-conditional effects, not uniform ones.

**Caveats to surface.** Same identification structure as sepsis: total
effects identified by randomization; mediated portion relies on
sequential-ignorability assumptions. The per-model heterogeneity is
larger in this task, so pooled mediation estimates should be reported
alongside per-model results rather than as standalone summaries.
Bootstrap inference on per-organization indirect effects controls the
false discovery rate at 0.05 across the five tests via Benjamini-
Hochberg adjustment.